- **Sell In:** Sell-in refers to sales from a manufacturer → distributor/retailer.
- **Sell Out:** Sell-out refers to sales from a retailer → end customer.

In [78]:
import warnings
warnings.filterwarnings("ignore")

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, kurtosis, skew
from scipy.signal import correlate

import helper_fn
import importlib
importlib.reload(helper_fn)


<module 'helper_fn' from '/home/asad/Desktop/footfall_explorer/src/footfall_explorer/notebooks/original_data_work/helper_fn.py'>

In [80]:
base_dir = "../../../../data/France/"

PROVIDERS = ['Bimedia', 'Devlyx', 'Logista']
MONTHS = ['Jan', 'Feb', 'March']

In [81]:
df_sku = pd.read_excel(os.path.join(base_dir, "Account and SKU Master Data", "FR_SKU Extract.xlsx")) 
df_outlets = pd.read_excel(os.path.join(base_dir, "Account and SKU Master Data", "FR_Outlets Extract_Paris.xlsx"))

## Load Sell In Data for Jan, Feb, Mar
sell_in_jan = pd.read_excel(os.path.join(base_dir, "Logista_Paris_Sell In_Jan_Feb_Mar", "Logista_Paris_Sell In_Jan.xlsx"))
sell_in_feb = pd.read_excel(os.path.join(base_dir, "Logista_Paris_Sell In_Jan_Feb_Mar", "Logista_Paris_Sell In_Feb.xlsx"))
sell_in_mar = pd.read_excel(os.path.join(base_dir, "Logista_Paris_Sell In_Jan_Feb_Mar", "Logista_Paris_Sell In_Mar.xlsx"))

## Load Sell Out Data.
sell_out_data = helper_fn.load_sellout_data(os.path.join(base_dir, "Sell out data"))

In [82]:
sell_out_data.keys()

dict_keys([('Bimedia', 'Feb'), ('Bimedia', 'Jan'), ('Bimedia', 'March'), ('Devlyx', 'March'), ('Devlyx', 'Jan'), ('Devlyx', 'Feb'), ('Logista', 'March'), ('Logista', 'Feb'), ('Logista', 'Jan')])

### Column Standardization and Cleaning

In [83]:
BASE_COL = [c for c in sell_in_feb.columns if 'BASE UNIT' in c][0]


In [84]:
sell_in_jan = helper_fn.clean_sellin(sell_in_jan, 'Jan', BASE_COL)

In [85]:

sell_in_feb = helper_fn.clean_sellin(sell_in_feb, 'Feb', BASE_COL)
sell_in_mar = helper_fn.clean_sellin(sell_in_mar, 'Mar', BASE_COL)

In [86]:
sell_in_jan.head()

,sales_date_raw,pos_code,outlet_sf_id,sku_code,sku_sf_id,sku_record,delivery_date_raw,base_unit,outer_unit,date,delivery_date,delivery_lag,month,dow,week,day
0,20260105,300003,0011t000011b5fiAAA,86195,a0U3W000002A7QeUAK,Trade SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5
1,20260105,300003,0011t000011b5fiAAA,86196,a0U3W000002A7QtUAK,Trade SKU,20260106,15,3,2026-01-05,2026-01-06,1,Jan,Monday,2,5
2,20260105,300003,0011t000011b5fiAAA,87571,a0UJz000005H9AVMA0,Trade SKU,20260106,10,2,2026-01-05,2026-01-06,1,Jan,Monday,2,5
3,20260105,300003,0011t000011b5fiAAA,87866,a0UJz000009Ce1KMAS,Manufacturing SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5
4,20260105,300003,0011t000011b5fiAAA,87884,a0UJz000009CevlMAC,Manufacturing SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5


In [ ]:
# sell_in_feb.rename(columns={" SKU SF ID": "sku_sf_id", "RecordType": "sku_record"}, inplace=True)
sell_in_feb.head()

,sales_date_raw,pos_code,outlet_sf_id,sku_code,sku_sf_id,sku_record,delivery_date_raw,base_unit,outer_unit,date,delivery_date,delivery_lag,month,dow,week,day
0,20260202,300054,0011t000011b36KAAQ,86195,a0U3W000002A7QeUAK,Trade SKU,20260202,5,1,2026-02-02,2026-02-02,0,Feb,Monday,6,2
1,20260202,300178,0011t000011b3BSAAY,86195,a0U3W000002A7QeUAK,Trade SKU,20260202,5,1,2026-02-02,2026-02-02,0,Feb,Monday,6,2
2,20260202,300178,0011t000011b3BSAAY,86196,a0U3W000002A7QtUAK,Trade SKU,20260202,5,1,2026-02-02,2026-02-02,0,Feb,Monday,6,2
3,20260202,300178,0011t000011b3BSAAY,87571,a0UJz000005H9AVMA0,Trade SKU,20260202,5,1,2026-02-02,2026-02-02,0,Feb,Monday,6,2
4,20260202,300178,0011t000011b3BSAAY,87812,a0UJz000007USXVMA4,Manufacturing SKU,20260202,6,1,2026-02-02,2026-02-02,0,Feb,Monday,6,2


In [ ]:
# sell_in_mar.rename(columns={"SKU SF Id": "sku_sf_id", "RecordType": "sku_record"}, inplace=True)
sell_in_mar.head()

,sales_date_raw,pos_code,outlet_sf_id,sku_code,sku_sf_id,sku_record,delivery_date_raw,base_unit,outer_unit,date,delivery_date,delivery_lag,month,dow,week,day
0,20260302,300054,0011t000011b36KAAQ,86195,a0U3W000002A7QeUAK,Trade SKU,20260302,5,1,2026-03-02,2026-03-02,0,Mar,Monday,10,2
1,20260302,300054,0011t000011b36KAAQ,86196,a0U3W000002A7QtUAK,Trade SKU,20260302,5,1,2026-03-02,2026-03-02,0,Mar,Monday,10,2
2,20260302,300054,0011t000011b36KAAQ,87571,a0UJz000005H9AVMA0,Trade SKU,20260302,5,1,2026-03-02,2026-03-02,0,Mar,Monday,10,2
3,20260302,300054,0011t000011b36KAAQ,87813,a0UJz00000BJO3WMAX,Trade SKU,20260302,6,1,2026-03-02,2026-03-02,0,Mar,Monday,10,2
4,20260302,300178,0011t000011b3BSAAY,86195,a0U3W000002A7QeUAK,Trade SKU,20260302,10,2,2026-03-02,2026-03-02,0,Mar,Monday,10,2


In [97]:
sell_in = pd.concat([sell_in_jan, sell_in_feb, sell_in_mar], ignore_index=True)

In [99]:
sell_in_jan.shape, sell_in_feb.shape, sell_in_mar.shape, sell_in.shape

((41727, 16), (40763, 16), (39817, 16), (122307, 16))

In [101]:
sell_in.to_csv(os.path.join(base_dir, "processed_data", "sell_in.csv"), index=False)

### Sell Out

In [102]:
def clean_sellout(df, source, month_label):
    df = df.copy()
    # Bimedia columns: Sales Date, POS, Outlet SF ID, City, SKU, SKU SF ID, RecordType, Volume in Unit, Volume in Packs
    # Devlyx / Logista sell-out may differ – normalise to a common schema
    col_map = {}
    for c in df.columns:
        cl = c.lower().strip()
        if 'sales date'   in cl: col_map[c] = 'sales_date_raw'
        elif 'outlet sf'  in cl: col_map[c] = 'outlet_sf_id'
        elif cl in ('pos','point of sales','pos code','point of sales code'): col_map[c] = 'pos_code'
        elif 'sku sf'     in cl: col_map[c] = 'sku_sf_id'
        elif 'recordtype' in cl or 'sku record' in cl: col_map[c] = 'sku_record'
        elif 'volume in unit' in cl or cl == 'qty' or cl == 'units': col_map[c] = 'volume_unit'
        elif 'volume in pack' in cl or 'packs' in cl: col_map[c] = 'volume_packs'
        elif cl == 'sku':        col_map[c] = 'sku_code'
        elif cl == 'city':       col_map[c] = 'city'
    df.rename(columns=col_map, inplace=True)

    # Ensure required cols exist
    for col in ['pos_code', 'sku_code', 'city', 'volume_packs']:
        if col not in df.columns:
            df[col] = np.nan

    df['date']   = pd.to_datetime(df['sales_date_raw'].astype(str), format='%Y%m%d')
    df['month']  = month_label
    df['source'] = source
    df['dow']    = df['date'].dt.day_name()
    df['week']   = df['date'].dt.isocalendar().week.astype(int)
    df['day']    = df['date'].dt.day

    # Sentinel 0 → NaN
    df['sku_sf_id']  = df['sku_sf_id'].replace(0, np.nan)
    df['sku_record'] = df['sku_record'].replace(0, np.nan)
    return df

In [103]:
sell_out_clean = {}
for (source, month), df in sell_out_data.items():
    sell_out_clean[(source, month)] = clean_sellout(df, source, month)

# Stack all sell-out into one DataFrame
sell_out_all = pd.concat(sell_out_clean.values(), ignore_index=True)

In [107]:
sell_out_all.to_csv(os.path.join(base_dir, "processed_data", "sell_out.csv"), index=False)

## Basic Statistics

In [108]:
sell_in.head()

,sales_date_raw,pos_code,outlet_sf_id,sku_code,sku_sf_id,sku_record,delivery_date_raw,base_unit,outer_unit,date,delivery_date,delivery_lag,month,dow,week,day
0,20260105,300003,0011t000011b5fiAAA,86195,a0U3W000002A7QeUAK,Trade SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5
1,20260105,300003,0011t000011b5fiAAA,86196,a0U3W000002A7QtUAK,Trade SKU,20260106,15,3,2026-01-05,2026-01-06,1,Jan,Monday,2,5
2,20260105,300003,0011t000011b5fiAAA,87571,a0UJz000005H9AVMA0,Trade SKU,20260106,10,2,2026-01-05,2026-01-06,1,Jan,Monday,2,5
3,20260105,300003,0011t000011b5fiAAA,87866,a0UJz000009Ce1KMAS,Manufacturing SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5
4,20260105,300003,0011t000011b5fiAAA,87884,a0UJz000009CevlMAC,Manufacturing SKU,20260106,5,1,2026-01-05,2026-01-06,1,Jan,Monday,2,5


In [111]:
sell_out_all.head()

,sales_date_raw,pos_code,outlet_sf_id,city,sku_code,sku_sf_id,sku_record,volume_unit,volume_packs,date,month,source,dow,week,day
0,20260201,576579,0011t000011b5rcAAA,PARIS,87812,a0UJz000007USXVMA4,Manufacturing SKU,1,1.0,2026-02-01,Feb,Bimedia,Sunday,5,1
1,20260201,576579,0011t000011b5rcAAA,PARIS,87865,a0UJz000009CfTdMAK,Manufacturing SKU,1,1.0,2026-02-01,Feb,Bimedia,Sunday,5,1
2,20260201,576579,0011t000011b5rcAAA,PARIS,87867,a0UJz000009CfrpMAC,Manufacturing SKU,1,1.0,2026-02-01,Feb,Bimedia,Sunday,5,1
3,20260201,307057,0011t000011bCLOAA2,PARIS,86196,a0U3W000002A7QtUAK,Trade SKU,80,4.0,2026-02-01,Feb,Bimedia,Sunday,5,1
4,20260201,307057,0011t000011bCLOAA2,PARIS,87635,a0UJz00000C50QHMAZ,Trade SKU,1,1.0,2026-02-01,Feb,Bimedia,Sunday,5,1


In [109]:
df_outlets.head()

,Store Participant Code,Outlet Name,Outlet Salesforce Id,Status,City,State,Latitude,Longitude,Ownership Type,Territory Id,Unnamed: 10,Terr Name,Area,EM Region Name,Division
0,331630,331630 - POURQUOI PAS,0011t000011b2TVAAY,Active,PARIS,ILE-DE-FRANCE,48.82327,2.35787,COCO,a0D1t000005iLAAEA2,a0D1t000005iLAAEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_10,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
1,308125,308125 - ROND POINT TAVERNE,0011t000011b31tAAA,Active,PARIS,ILE-DE-FRANCE,48.83869,2.35088,COCO,a0D1t000005iLAAEA2,a0D1t000005iLAAEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_10,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
2,303025,303025 - FONTENOY (LE),0011t000011b3DNAAY,Active,PARIS,ILE-DE-FRANCE,48.89117,2.34725,COCO,a0D1t000005iLAnEAM,a0D1t000005iLAnEAM,France_FRDiv_Reg1_AREA_A_SECTEUR_A_06,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
3,305109,305109 - LUTETIA (LE),0011t000011b2EbAAI,Active,PARIS,ILE-DE-FRANCE,48.87599,2.36807,COCO,a0D1t000005iLBVEA2,a0D1t000005iLBVEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_08,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
4,304122,304122 - UNIVERSITE (TABAC DE L'),0011t000011b2BNAAY,Active,PARIS,ILE-DE-FRANCE,48.86107,2.30530,COCO,a0D1t000005iLAGEA2,a0D1t000005iLAGEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_01,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv


In [146]:
sell_in[sell_in["sku_code"] == 99999].shape

(21801, 16)

In [122]:
sell_in.shape, sell_out_all.shape, df_outlets.shape

((122307, 16), (929460, 15), (958, 15))

In [148]:
df_outlets['Status'].value_counts()
# sell_in[sell_in['Status']]

Status
Active                     915
Temporarily Deactivated     43
Name: count, dtype: int64

In [115]:
total_unique_shops_in_sell_in = sell_in['outlet_sf_id'].nunique()
total_unique_shops_in_sell_out = sell_out_all['outlet_sf_id'].nunique()

print(f"Total Unique shops in master outlet data: {df_outlets['Store Participant Code'].nunique()}")
print(f"Total unique shops in Sell-In data: {total_unique_shops_in_sell_in}")
print(f"Total unique shops in Sell-Out data: {total_unique_shops_in_sell_out}")

Total Unique shops in master outlet data: 801
Total unique shops in Sell-In data: 650
Total unique shops in Sell-Out data: 1063


In [123]:
# ── Work on copies so originals are untouched ────────────────────────────────
si = sell_in.copy()
so = sell_out_all.copy()

# ── Exclude dummy SKU 99999 from all statistics ──────────────────────────────
# si = si[si['sku_code'] != 99999]
# so = so[so['sku_code'] != 99999]

# ── Normalise month label (sell-out has 'March', sell-in has 'Mar') ──────────
so['month_s'] = so['month'].map({'Jan': 'Jan', 'Feb': 'Feb', 'March': 'Mar'}).fillna(so['month'])
si['month_s'] = si['month']

MONTH_ORDER = ['Jan', 'Feb', 'Mar']

# ── Week and day helpers ─────────────────────────────────────────────────────
si['week_label'] = si['date'].dt.to_period('W')
so['week_label'] = so['date'].dt.to_period('W')

si['year_week'] = si['date'].dt.isocalendar().year.astype(str) + '-W' + \
                  si['date'].dt.isocalendar().week.astype(str).str.zfill(2)
so['year_week'] = so['date'].dt.isocalendar().year.astype(str) + '-W' + \
                  so['date'].dt.isocalendar().week.astype(str).str.zfill(2)

print(f"Working sell-in rows  (excl. dummy): {len(si):,}")
print(f"Working sell-out rows (excl. dummy): {len(so):,}")
print(f"\nSell-in date range  : {si['date'].min().date()} → {si['date'].max().date()}")
print(f"Sell-out date range : {so['date'].min().date()} → {so['date'].max().date()}")

Working sell-in rows  (excl. dummy): 122,307
Working sell-out rows (excl. dummy): 929,460

Sell-in date range  : 2026-01-05 → 2026-03-30
Sell-out date range : 2025-12-29 → 2026-03-31


In [128]:
# ─────────────────────────────────────────────────────────────────────────────
# STAT 1 — SHOP UNIVERSE
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  STAT 1 — SHOP UNIVERSE")
print("=" * 65)

si_shops_total = si['outlet_sf_id'].nunique()
so_shops_total = so['outlet_sf_id'].nunique()

si_shops_by_month = si.groupby('month_s')['outlet_sf_id'].nunique().reindex(MONTH_ORDER)
so_shops_by_month = so.groupby('month_s')['outlet_sf_id'].nunique().reindex(MONTH_ORDER)

print(f"\n{'':4}{'SOURCE':<22}{'TOTAL UNIQUE SHOPS':>20}")
print(f"{'':4}{'-'*42}")
print(f"{'':4}{'Sell-In (Logista)':<22}{si_shops_total:>20,}")
print(f"{'':4}{'Sell-Out (all sources)':<22}{so_shops_total:>20,}")

print(f"\n  Sell-Out breakdown by source:")
for src in so['source'].unique():
    n = so[so['source'] == src]['outlet_sf_id'].nunique()
    print(f"    {src:<18}: {n:,} shops")

print(f"\n  Shops per month:")
print(f"\n  {'MONTH':<8}{'Sell-In':>12}{'Sell-Out':>12}")
print(f"  {'-'*32}")
for m in MONTH_ORDER:
    si_n = si_shops_by_month.get(m, 0)
    so_n = so_shops_by_month.get(m, 0)
    print(f"  {m:<8}{int(si_n):>12,}{int(so_n):>12,}")




  STAT 1 — SHOP UNIVERSE

    SOURCE                  TOTAL UNIQUE SHOPS
    ------------------------------------------
    Sell-In (Logista)                      650
    Sell-Out (all sources)               1,063

  Sell-Out breakdown by source:
    Bimedia           : 145 shops
    Devlyx            : 14 shops
    Logista           : 906 shops

  Shops per month:

  MONTH        Sell-In    Sell-Out
  --------------------------------
  Jan              648         601
  Feb              648         603
  Mar              649       1,061


In [126]:
# ─────────────────────────────────────────────────────────────────────────────
# STAT 2 — MATCHING SHOPS: SI ∩ SO
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  STAT 2 — MATCHING SHOPS (Sell-In ∩ Sell-Out)")
print("=" * 65)

si_set = set(si['outlet_sf_id'].dropna())
so_set = set(so['outlet_sf_id'].dropna())

both        = si_set & so_set
only_si     = si_set - so_set
only_so     = so_set - si_set

print(f"\n  Shops in BOTH sell-in and sell-out  : {len(both):>6,}  ← matched / gold set")
print(f"  Shops ONLY in sell-in               : {len(only_si):>6,}  ← no consumer signal")
print(f"  Shops ONLY in sell-out              : {len(only_so):>6,}  ← anomalous")

print(f"\n  Match rate (SO shops found in SI)   : {len(both)/len(so_set)*100:.1f}%")
print(f"  Match rate (SI shops found in SO)   : {len(both)/len(si_set)*100:.1f}%")

# Per-month overlap
print(f"\n  Month-by-month overlap:")
print(f"\n  {'MONTH':<8}{'SI shops':>10}{'SO shops':>10}{'In both':>10}{'Only SI':>10}{'Only SO':>10}")
print(f"  {'-'*58}")
for m in MONTH_ORDER:
    si_m = set(si[si['month_s'] == m]['outlet_sf_id'].dropna())
    so_m = set(so[so['month_s'] == m]['outlet_sf_id'].dropna())
    b = si_m & so_m
    print(f"  {m:<8}{len(si_m):>10,}{len(so_m):>10,}{len(b):>10,}"
          f"{len(si_m-so_m):>10,}{len(so_m-si_m):>10,}")

# By sell-out source
print(f"\n  Overlap by sell-out source:")
print(f"\n  {'SOURCE':<14}{'SO shops':>10}{'Found in SI':>14}{'Match %':>10}")
print(f"  {'-'*50}")
for src in sorted(so['source'].unique()):
    src_set = set(so[so['source'] == src]['outlet_sf_id'].dropna())
    overlap = src_set & si_set
    pct     = len(overlap) / len(src_set) * 100 if src_set else 0
    print(f"  {src:<14}{len(src_set):>10,}{len(overlap):>14,}{pct:>9.1f}%")

  STAT 2 — MATCHING SHOPS (Sell-In ∩ Sell-Out)

  Shops in BOTH sell-in and sell-out  :    602  ← matched / gold set
  Shops ONLY in sell-in               :     48  ← no consumer signal
  Shops ONLY in sell-out              :    461  ← anomalous

  Match rate (SO shops found in SI)   : 56.6%
  Match rate (SI shops found in SO)   : 92.6%

  Month-by-month overlap:

  MONTH     SI shops  SO shops   In both   Only SI   Only SO
  ----------------------------------------------------------
  Jan            648       601       600        48         1
  Feb            648       603       600        48         3
  Mar            649     1,061       601        48       460

  Overlap by sell-out source:

  SOURCE          SO shops   Found in SI   Match %
  --------------------------------------------------
  Bimedia              145           144     99.3%
  Devlyx                14             7     50.0%
  Logista              906           453     50.0%


In [131]:
si_total_days = (si['date'].max() - si['date'].min()).days + 1
so_total_days = (so['date'].max() - so['date'].min()).days + 1
si_working_days = si['date'].nunique()   # only weekdays have deliveries
so_calendar_days = so['date'].nunique()

print(f"\n  Calendar span:")
print(f"    Sell-in  : {si_total_days} calendar days  | {si_working_days} unique transaction days (weekdays only)")
print(f"    Sell-out : {so_total_days} calendar days  | {so_calendar_days} unique transaction days (incl. weekends)")




  Calendar span:
    Sell-in  : 85 calendar days  | 61 unique transaction days (weekdays only)
    Sell-out : 93 calendar days  | 93 unique transaction days (incl. weekends)


In [ ]:
si_dates_per_outlet = si.groupby('outlet_sf_id')['date'].nunique()
so_dates_per_outlet = so.groupby('outlet_sf_id')['date'].nunique()


In [136]:
print("Sell In Daily transactions per outlet:")
si_dates_per_outlet.describe()

Sell In Daily transactions per outlet:


count    650.000000
mean       8.130769
std        3.192799
min        2.000000
25%        6.000000
50%        7.000000
75%        9.000000
max       26.000000
Name: date, dtype: float64

In [140]:
print("Sell out Daily transactions per outlet:")
so_dates_per_outlet.describe()


Sell out Daily transactions per outlet:


count    1063.000000
mean       58.171214
std        27.262980
min         1.000000
25%        30.500000
50%        74.000000
75%        85.000000
max        90.000000
Name: date, dtype: float64

### Weekly Frequency

In [141]:
si_weeks_per_outlet = si.groupby('outlet_sf_id')['week_label'].nunique()
so_weeks_per_outlet = so.groupby('outlet_sf_id')['week_label'].nunique()

In [143]:
si_weeks_per_outlet.describe()

count    650.000000
mean       7.069231
std        2.004573
min        1.000000
25%        6.000000
50%        6.000000
75%        8.000000
max       13.000000
Name: week_label, dtype: float64

In [15]:
df_outlets.head()

,Store Participant Code,Outlet Name,Outlet Salesforce Id,Status,City,State,Latitude,Longitude,Ownership Type,Territory Id,Unnamed: 10,Terr Name,Area,EM Region Name,Division
0,331630,331630 - POURQUOI PAS,0011t000011b2TVAAY,Active,PARIS,ILE-DE-FRANCE,48.82327,2.35787,COCO,a0D1t000005iLAAEA2,a0D1t000005iLAAEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_10,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
1,308125,308125 - ROND POINT TAVERNE,0011t000011b31tAAA,Active,PARIS,ILE-DE-FRANCE,48.83869,2.35088,COCO,a0D1t000005iLAAEA2,a0D1t000005iLAAEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_10,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
2,303025,303025 - FONTENOY (LE),0011t000011b3DNAAY,Active,PARIS,ILE-DE-FRANCE,48.89117,2.34725,COCO,a0D1t000005iLAnEAM,a0D1t000005iLAnEAM,France_FRDiv_Reg1_AREA_A_SECTEUR_A_06,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
3,305109,305109 - LUTETIA (LE),0011t000011b2EbAAI,Active,PARIS,ILE-DE-FRANCE,48.87599,2.36807,COCO,a0D1t000005iLBVEA2,a0D1t000005iLBVEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_08,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
4,304122,304122 - UNIVERSITE (TABAC DE L'),0011t000011b2BNAAY,Active,PARIS,ILE-DE-FRANCE,48.86107,2.30530,COCO,a0D1t000005iLAGEA2,a0D1t000005iLAGEA2,France_FRDiv_Reg1_AREA_A_SECTEUR_A_01,FR_France_FRDiv_Reg1_AREA_A,FR_France_FRDiv_Reg1,FR_France_FRDiv


In [ ]:
## SKU_Code is unique in df_sku. 
## Outlet Salesforce Id is unique in df_outlets.
## **Store Participant Code**, we have a 0 and it is repeated 158 times, and all the other codes are unique.
# Also all those stores with Store Participant Code == 0, has no Lat and Lon values.    

print("Number of unique SKU Codes: ", df_sku.shape)
print(df_outlets["Store Participant Code"].nunique(), df_outlets.shape)

Number of unique SKU Codes:  (4134, 10)
801 (958, 15)


,Store Participant Code,Outlet Name,Outlet Salesforce Id,Status,City,State,Latitude,Longitude,Ownership Type,Territory Id,Unnamed: 10,Terr Name,Area,EM Region Name,Division
36,0,302080 - RELAY P-LYON VOIE A SNCF,0011t000011aybIAAQ,Temporarily Deactivated,PARIS,ILE-DE-FRANCE,0.0,0.0,COCO,a0D1t000005iLBYEA2,a0D1t000005iLBYEA2,France_FRDiv_Reg1_AREA_C_SECTEUR_C_24,FR_France_FRDiv_Reg1_AREA_C,FR_France_FRDiv_Reg1,FR_France_FRDiv
39,0,303006 - RELAY NORD ZONE ECHANGE NIV-1,0011t000011ayesAAA,Temporarily Deactivated,PARIS,ILE-DE-FRANCE,0.0,0.0,COCO,a0D1t000005iLBYEA2,a0D1t000005iLBYEA2,France_FRDiv_Reg1_AREA_C_SECTEUR_C_24,FR_France_FRDiv_Reg1_AREA_C,FR_France_FRDiv_Reg1,FR_France_FRDiv
43,0,300002 - RELAY H TROUVE (TABAC),0011t000011ayiTAAQ,Temporarily Deactivated,PARIS,ILE-DE-FRANCE,0.0,0.0,COCO,a0D1t000005iLBYEA2,a0D1t000005iLBYEA2,France_FRDiv_Reg1_AREA_C_SECTEUR_C_24,FR_France_FRDiv_Reg1_AREA_C,FR_France_FRDiv_Reg1,FR_France_FRDiv
44,0,303197 - RELAY NORD ACCES GAL NIV-1 SNCF,0011t000011aylvAAA,Temporarily Deactivated,PARIS,ILE-DE-FRANCE,0.0,0.0,COCO,a0DJz000000rRdBMAU,a0DJz000000rRdBMAU,France_FRDiv_Reg1_AREA_OMNI_A_SECTEUR_OMNI_7,FR_France_FRDiv_Reg1_AREA_OMNI_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
45,0,300001 - RELAY METRO METEOR ST LAZARE,0011t000011aysjAAA,Temporarily Deactivated,PARIS,ILE-DE-FRANCE,0.0,0.0,COCO,a0DJz000000rRi1MAE,a0DJz000000rRi1MAE,France_FRDiv_Reg1_AREA_OMNI_A_SECTEUR_OMNI_9,FR_France_FRDiv_Reg1_AREA_OMNI_A,FR_France_FRDiv_Reg1,FR_France_FRDiv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,0,One Clop,0011t000011bDlbAAE,Active,paris,ILE-DE-FRANCE,0.0,0.0,Independent,a0D1t000005iLBdEAM,a0D1t000005iLBdEAM,France_FRDiv_Reg3_AREA_1_SECTEUR_9,FR_France_FRDiv_Reg3_AREA_1,FR_France_FRDiv_Reg3,FR_France_FRDiv
952,0,Vapostore Paris 16,0011t000011bDvAAAU,Active,Paris,ILE-DE-FRANCE,0.0,0.0,Independent,a0D1t000005iLBdEAM,a0D1t000005iLBdEAM,France_FRDiv_Reg3_AREA_1_SECTEUR_9,FR_France_FRDiv_Reg3_AREA_1,FR_France_FRDiv_Reg3,FR_France_FRDiv
953,0,le kiosque presse - Nahid Paul,0011t000011bDvoAAE,Active,paris,ILE-DE-FRANCE,0.0,0.0,Independent,a0D1t000005iLBdEAM,a0D1t000005iLBdEAM,France_FRDiv_Reg3_AREA_1_SECTEUR_9,FR_France_FRDiv_Reg3_AREA_1,FR_France_FRDiv_Reg3,FR_France_FRDiv
954,0,Kitclope,0011t000011bDvpAAE,Active,paris,ILE-DE-FRANCE,0.0,0.0,Independent,a0D1t000005iLBdEAM,a0D1t000005iLBdEAM,France_FRDiv_Reg3_AREA_1_SECTEUR_9,FR_France_FRDiv_Reg3_AREA_1,FR_France_FRDiv_Reg3,FR_France_FRDiv


In [ ]:

# total dokanen kitne hain. 

# kitne ka sell in and sell out data mojood ha. 

# sell in and sell out weekly horha ha ya kuch or ha. 

# sell in and sell out kitne frequency se araha ha. 

In [ ]:
### Plotting Styles 

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#161b27',
    'axes.edgecolor':    '#2a3040',
    'axes.labelcolor':   '#c9cdd4',
    'axes.grid':         True,
    'grid.color':        '#1e2535',
    'grid.linewidth':    0.6,
    'xtick.color':       '#7a8494',
    'ytick.color':       '#7a8494',
    'text.color':        '#c9cdd4',
    'figure.titlesize':  15,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'legend.framealpha': 0.15,
    'legend.edgecolor':  '#2a3040',
    'font.family':       'DejaVu Sans',
    'lines.linewidth':   1.8,
})

# Palette
C_TEAL   = '#6ee7b7'
C_PINK   = '#f472b6'
C_AMBER  = '#fbbf24'
C_INDIGO = '#818cf8'
C_ORANGE = '#fb923c'
C_RED    = '#f87171'
C_GRAY   = '#4b5563'

PALETTE = [C_TEAL, C_PINK, C_AMBER, C_INDIGO, C_ORANGE, C_RED, '#a78bfa', '#34d399']

In [ ]:
sell_out_data.keys()

dict_keys([('Bimedia', 'Feb'), ('Bimedia', 'Jan'), ('Bimedia', 'March'), ('Devlyx', 'March'), ('Devlyx', 'Jan'), ('Devlyx', 'Feb'), ('Logista', 'March'), ('Logista', 'Feb'), ('Logista', 'Jan')])